## Grid Search for Analyzer Settings

Re-runs PubTrends clustering with different weights for the following measures:
 * bibliographic coupling (BC)
 * co-citations (CC)
 * direct citations (DC)
 * text citations (TC)

In [2]:
import json
import itertools
import logging
import os

from sklearn.metrics.cluster import adjusted_mutual_info_score, v_measure_score

from pysrc.papers.config import AnalyzerSettings

from utils.analysis import rebuild_similarity_graph, get_direct_references_subgraph, \
                           recalculate_topic_analysis, align_clusterings_for_sklearn
from utils.io import reload_exported_analyzer, load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [3]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [4]:
# Optionally full path can be provided
OUTPUT_NAME = 'grid_search_2021-04-11_22_30.json'

In [5]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [9]:
def run_grid_search(analyzer, subgraph, ground_truth, 
                    metrics, target_metric,
                    baseline_params, param_names, param_range):
    """
    In order to reduce number of options:
        1. Fix one of the parameters as 1, change others. 
        2. Remove fixed parameter by setting it to 0.
        3. Go to step 1 for remaining subset.
    
    Example for (cocitation, bibcoupling, citation, text_citation), ch = parameter is changed via param_range: 
    
        (1, ch, ch, ch)
        (0,  1, ch, ch)
        (0,  0,  1, ch)
        (0,  0,  0,  1)
    """
    # Accumulate grid results for all hierarchy levels
    grid_results = []
    best_results = {k: (0, None) for k in ground_truth.keys()}
    
    # Iterate the parameter to be fixed
    for i, param in enumerate(param_names):
        const_params = baseline_params.copy()
        param_grid = {}
        
        # Setup ranges for all parameters
        for j, other_param in enumerate(param_names):
            if j <= i:
                # Add parameter weight to constants as 0 or 1
                const_params[other_param] = int(j == i)
            else:
                # Add parameter to grid
                param_grid[other_param] = param_range
    
        # Iterate over obtained param grid
        changing_param_names = param_grid.keys()
        for param_values in itertools.product(*param_grid.values()):
            # Unfold grid parameters and add constant ones
            params = {k: v for k, v in zip(changing_param_names, param_values)}
            for k, v in const_params.items():
                params[k] = v

            # Apply settings to the analyzer and recalculate the partition
            settings = AnalyzerSettings(**params)
            soi = settings_of_interest(settings)
            partition = recalculate_topic_analysis(analyzer, 
                                                   graph=subgraph,
                                                   settings=settings)
            
            grid_results.append({
                'soi': list(soi),
                'partition': partition
            })
            
            # Iterate over hierarchy levels to avoid re-calculating same clustering for different GTs
            for level, ground_truth_partition in ground_truth.items():
                labels_true, labels_pred = align_clusterings_for_sklearn(partition, ground_truth_partition)

                # Save partition & iterate over different metrics
                result = {'partition': partition}
                for metric in metrics:
                    result[metric.__name__] = metric(labels_true, labels_pred)

                new_best = False
                best_score = best_results[level][0]
                if result[target_metric] > best_score:
                    best_results[level] = (result[target_metric], soi)
                    new_best = True

            print('.', end='')
    print('\n')
    
    return grid_results, best_results

In [10]:
param_range = [0, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
# param_range = [0, 1, 2]  # For quick testing

# Order matters when choosing the parameter to be fixed.
# Current order reflects intuition about usefulness of the parameters
param_names = ['SIMILARITY_COCITATION',
               'SIMILARITY_BIBLIOGRAPHIC_COUPLING',
               'SIMILARITY_CITATION',
               'SIMILARITY_TEXT_CITATION']

In [11]:
# Turn off all post-processing of clustering
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [12]:
metrics = [adjusted_mutual_info_score, v_measure_score, pd_score, reg_v_score]
target_metric = pd_score.__name__

In [13]:
results = []

for pmid in get_review_pmids():
    clustering = load_clustering(pmid)
    analyzer = load_analyzer(pmid)
    logging.info(f'{pmid} - loaded clustering and analyzer')
    rebuild_similarity_graph(analyzer, min_cocitation=2)
    logging.info(f'{pmid} - rebuilt similarity graph with scaling')
    
    # Pre-calculate all hierarchy levels before grid search to avoid re-calculation of clusterings
    ground_truth = {}
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[level] = preprocess_clustering(clustering, level, 
                                                    include_box_sections=False,
                                                    uniqueness_method='unique_only')
    logging.info(f'{pmid} - calculated ground truth for all hierarchy levels')
    
    subgraph = get_direct_references_subgraph(analyzer, pmid)
    logging.info(f'{pmid} - running grid search')
    
    grid_results, best_results = run_grid_search(analyzer, subgraph, ground_truth, 
                                                 metrics, target_metric,
                                                 baseline_params, param_names, param_range)
    
    for level, (score, soi) in best_results.items():
        logging.info(f'{pmid} - level {level} - best score: {score}, best params: {soi}')
    
    results.append({
        'pmid': pmid,
        'results': grid_results
    })
    
    logging.info(f'{pmid} - done')

2021-04-11 20:34:48,524 INFO: 26580716 - loaded clustering and analyzer
2021-04-11 20:34:48,530 INFO: Building corpus from 313 papers
2021-04-11 20:34:59,792 INFO: 26580716 - rebuilt similarity graph with scaling
2021-04-11 20:34:59,794 INFO: 26580716 - calculated ground truth for all hierarchy levels
2021-04-11 20:34:59,796 INFO: 26580716 - running grid search


........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

2021-04-11 20:43:31,872 INFO: 26580716 - level 1 - best score: 0.36140967090745213, best params: (1, 0.125, 1, 16)
2021-04-11 20:43:31,874 INFO: 26580716 - level 2 - best score: 0.365334627602768, best params: (1, 0.125, 1, 16)
2021-04-11 20:43:31,875 INFO: 26580716 - done


.



2021-04-11 20:43:32,792 INFO: 26580717 - loaded clustering and analyzer
2021-04-11 20:43:32,800 INFO: Building corpus from 374 papers
2021-04-11 20:43:41,592 INFO: 26580717 - rebuilt similarity graph with scaling
2021-04-11 20:43:41,595 INFO: 26580717 - calculated ground truth for all hierarchy levels
2021-04-11 20:43:41,601 INFO: 26580717 - running grid search


...

.



2021-04-12 03:20:35,641 INFO: 32699292 - loaded clustering and analyzer
2021-04-12 03:20:35,648 INFO: Building corpus from 316 papers
2021-04-12 03:20:43,808 INFO: 32699292 - rebuilt similarity graph with scaling
2021-04-12 03:20:43,810 INFO: 32699292 - calculated ground truth for all hierarchy levels
2021-04-12 03:20:43,821 INFO: 32699292 - running grid search


........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

2021-04-12 03:26:20,892 INFO: 32699292 - level 1 - best score: 0.41784836204212855, best params: (1, 0.0625, 0, 8)
2021-04-12 03:26:20,893 INFO: 32699292 - level 2 - best score: 0.40810081655362446, best params: (0, 0, 0, 1)
2021-04-12 03:26:20,894 INFO: 32699292 - done


.



In [15]:
with open(OUTPUT_NAME, 'w') as f:
    json.dump(results, f)

In [16]:
print('Done')

Done
